# セル1：Google Driveをマウントし、必要ライブラリを入れます


In [ ]:
# セル1：Google Driveをマウントし、必要ライブラリを入れます
import os
import shutil

# 壊れた /content/drive がある場合はいったん削除
if os.path.exists("/content/drive") and not os.path.ismount("/content/drive"):
    shutil.rmtree("/content/drive", ignore_errors=True)

from google.colab import drive
drive.mount("/content/drive")

!apt-get -qq update > /dev/null
!apt-get -qq install -y fonts-noto-cjk libcairo2 libpango-1.0-0 libpangocairo-1.0-0 libgdk-pixbuf-2.0-0 libffi-dev shared-mime-info nodejs npm > /dev/null
!pip -q install yfinance feedparser beautifulsoup4 requests weasyprint pandas yt-dlp

print("準備完了：Driveマウントとライブラリ導入が終わりました。")


In [ ]:
# セル2：あなたの情報を入力します
# Gmailの通常パスワードではなく「アプリパスワード」を入れてください。
# 入力されたパスワードは画面に表示されません。

import getpass
import os
from datetime import datetime

DRIVE_ROOT = "/content/drive/MyDrive/ai-investment-agent"
REPORT_DIR = os.path.join(DRIVE_ROOT, "reports")
LOG_DIR = os.path.join(DRIVE_ROOT, "logs")
DATA_DIR = os.path.join(DRIVE_ROOT, "data")
ASSETS_DIR = os.path.join(DRIVE_ROOT, "assets")

for p in [DRIVE_ROOT, REPORT_DIR, LOG_DIR, DATA_DIR, ASSETS_DIR]:
    os.makedirs(p, exist_ok=True)

print("保存先:", DRIVE_ROOT)
print("表紙画像フォルダ:", ASSETS_DIR)

EMAIL_TO = input("レポートを受け取るメールアドレスを入力してください: ").strip()
SMTP_USER = input("送信用Gmailアドレスを入力してください: ").strip()
EMAIL_FROM = SMTP_USER
SMTP_PASSWORD = getpass.getpass("Gmailアプリパスワード16桁を入力してください: ").strip()

send_choice = input("PDFをメール送信しますか？ y/n [y]: ").strip().lower()
SEND_EMAIL = (send_choice != "n")
dry_choice = input("dry-runで実行しますか？ y/n [n]: ").strip().lower()
DRY_RUN = (dry_choice == "y")

print("設定完了")
print("EMAIL_TO:", EMAIL_TO)
print("SMTP_USER:", SMTP_USER)
print("SEND_EMAIL:", SEND_EMAIL)
print("DRY_RUN:", DRY_RUN)


In [ ]:
# セル3：字幕/文字起こし優先で発言抽出→品質チェック→PDF生成→(本番時のみ)メール送信
import os, re, json, logging, requests, smtplib, subprocess, tempfile, shutil
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub")
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
from datetime import datetime, timezone, timedelta
from email.message import EmailMessage
from email.utils import parsedate_to_datetime
import feedparser, yt_dlp, pandas as pd
from weasyprint import HTML

try:
    from faster_whisper import WhisperModel
except Exception:
    WhisperModel = None

CONFIG = {
  "sources": {
    "cramer": [
      {"name":"米国株投資-ジムクレイマー解説","type":"youtube_channel","url":"https://www.youtube.com/@DeepCramerJP/videos","source_group":"Jim Cramer","source_kind":"commentary","directness":"commentary_summary","speaker_label":"DeepCramerJP","enabled":True,"max_items":3},
      {"name":"CNBC Television","type":"youtube_channel","url":"https://www.youtube.com/@CNBCtelevision/videos","source_group":"Jim Cramer","source_kind":"official_media","directness":"direct_statement","speaker_label":"Jim Cramer/CNBC","enabled":True,"max_items":5,"filters":["Jim Cramer","Cramer","Mad Money"]},
      {"name":"Makabeeの米国株【ジム・クレイマー応援ch】","type":"youtube_channel","url":"https://www.youtube.com/@makabee7/videos","source_group":"Jim Cramer","source_kind":"commentary","directness":"commentary_summary","speaker_label":"Makabee","enabled":True,"max_items":3}
    ],
    "dalio": [{"name":"Principles by Ray Dalio","type":"youtube_channel","url":"https://www.youtube.com/@principlesbyraydalio/videos","source_group":"Ray Dalio","source_kind":"official","directness":"direct_statement","speaker_label":"Ray Dalio","enabled":True,"max_items":3}],
    "galloway": [{"name":"Pivot","type":"podcast","rss_url":"","podcast_search_query":"Pivot Kara Swisher Scott Galloway","source_group":"Scott Galloway / Pivot","source_kind":"podcast","directness":"podcast_discussion","speaker_label":"Pivot / Scott Galloway関連視点","enabled":True,"max_items":3}]
  },
  "collection": {"timezone":"Asia/Tokyo","mentioned_tickers_window_days":2, "fetch_window_days":2, "yt_retry_count":5, "yt_retry_wait_sec":10},
  "transcript": {"transcribe_audio_if_no_subtitle": True, "whisper_model":"small", "whisper_device":"cpu", "whisper_compute_type":"int8", "whisper_max_duration_sec":3600, "whisper_audio_dir":"/tmp/whisper_audio", "preferred_langs":["ja","en"]}
}

company_kana_map={"Amazon":"アマゾン","Apple":"アップル","NVIDIA":"エヌビディア","Palantir":"パランティア","Microsoft":"マイクロソフト","Alphabet":"アルファベット","Meta":"メタ","Tesla":"テスラ","Oracle":"オラクル","Broadcom":"ブロードコム","Arm":"アーム","Block":"ブロック","CrowdStrike":"クラウドストライク","RTX":"アールティーエックス","GE Vernova":"ジーイー・ベルノバ","Solstice Advanced Materials":"ソルスティス・アドバンスト・マテリアルズ","Dutch Bros":"ダッチ・ブロス","Corning":"コーニング","Shopify":"ショッピファイ","CVS Health":"シーブイエス・ヘルス"}
entity_map={"AAPL":"Apple","NVDA":"NVIDIA","PLTR":"Palantir","MSFT":"Microsoft","AMZN":"Amazon","GOOGL":"Alphabet","META":"Meta","TSLA":"Tesla","ORCL":"Oracle","AVGO":"Broadcom","ARM":"Arm","SHOP":"Shopify","CVS":"CVS Health","BTC":"Bitcoin","SPY":"S&P 500 ETF"}

JST=timezone(timedelta(hours=9)); today=datetime.now(JST).strftime("%Y-%m-%d")
base_name=f"us_stock_ai_report_{today}"; DEBUG_DIR=os.path.join(DRIVE_ROOT,"debug"); os.makedirs(DEBUG_DIR,exist_ok=True)
pdf_path=os.path.join(REPORT_DIR,f"{base_name}.pdf"); html_path=os.path.join(DEBUG_DIR,f"{base_name}_debug.html")
raw_json_path=os.path.join(DEBUG_DIR,f"raw_sources_{today}.json"); source_csv_path=os.path.join(DEBUG_DIR,f"source_check_{today}.csv")
mentions_csv_path=os.path.join(DEBUG_DIR,f"extracted_mentions_{today}.csv"); by_person_csv_path=os.path.join(DEBUG_DIR,f"mentioned_by_person_{today}.csv")
statement_csv_path=os.path.join(DEBUG_DIR,f"statement_units_{today}.csv"); quality_json_path=os.path.join(DEBUG_DIR,f"report_quality_check_{today}.json")
log_path=os.path.join(LOG_DIR,f"daily_{today}.log")
logger=logging.getLogger("daily_report"); logger.setLevel(logging.INFO); logger.handlers.clear()
for h in [logging.FileHandler(log_path,encoding="utf-8"),logging.StreamHandler()]: h.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")); logger.addHandler(h)
def log_check(n,v): logger.info(f"[CHECK] {n}: {v}"); print(f"[CHECK] {n}: {v}")

def is_within_window(published_at_str: str, window_days: int) -> bool:
    if not published_at_str:
        return False
    try:
        if re.match(r"^\d{8}$", published_at_str):
            dt=datetime.strptime(published_at_str, "%Y%m%d").replace(tzinfo=JST)
        else:
            dt=parsedate_to_datetime(published_at_str)
            if dt is None:
                return False
            if dt.tzinfo is None:
                dt=dt.replace(tzinfo=JST)
            else:
                dt=dt.astimezone(JST)
        return (datetime.now(JST)-dt) <= timedelta(days=window_days)
    except Exception:
        return False

def parse_caption(raw_text: str, content_type_hint: str = "") -> str:
    t=(raw_text or "").strip()
    if not t:
        return ""
    if t.startswith("{") or '"events"' in t:
        try:
            data=json.loads(t)
            parts=[]
            for ev in data.get("events",[]):
                for seg in ev.get("segs",[]) or []:
                    parts.append(seg.get("utf8",""))
            return re.sub(r"\s+", " ", "".join(parts)).strip()
        except Exception:
            return ""
    if "WEBVTT" in t or "-->" in t:
        lines=[]
        for ln in t.splitlines():
            s=ln.strip()
            if not s or s.startswith("WEBVTT") or "-->" in s or s.isdigit():
                continue
            lines.append(re.sub(r"<.*?>", "", s))
        return re.sub(r"\s+", " ", " ".join(lines)).strip()
    if re.search(r"\n\d+\n\d{2}:\d{2}:\d{2}", t):
        lines=[re.sub(r"<.*?>", "", ln.strip()) for ln in t.splitlines() if ln.strip() and not ln.strip().isdigit() and "-->" not in ln]
        return re.sub(r"\s+", " ", " ".join(lines)).strip()
    return re.sub(r"\s+", " ", t).strip()

subprocess.run(["python","-m","pip","install","-q","-U","yt-dlp"], check=False)
whisper_model=None
if CONFIG["transcript"]["transcribe_audio_if_no_subtitle"]:
    if WhisperModel is None:
        subprocess.run(["python","-m","pip","install","-q","faster-whisper"], check=True)
        from faster_whisper import WhisperModel
    # Hugging Face Hub 認証（任意：トークンがあれば使う、無くても動作する）
    try:
        hf_token=os.environ.get("HF_TOKEN","")
        if not hf_token:
            try:
                from google.colab import userdata; hf_token=userdata.get("HF_TOKEN") or ""
            except Exception: hf_token=""
        if hf_token: os.environ["HF_TOKEN"]=os.environ["HUGGING_FACE_HUB_TOKEN"]=hf_token; log_check("HF token loaded","yes")
        else: log_check("HF token loaded","no (using anonymous access)")
    except Exception as ex: log_check("HF token loaded", f"skipped ({ex})")
    whisper_model=WhisperModel(CONFIG["transcript"]["whisper_model"], device=CONFIG["transcript"]["whisper_device"], compute_type=CONFIG["transcript"]["whisper_compute_type"])

def transcribe_with_whisper(url: str, lang_hint: str = None) -> dict:
    out={"text":"","segments":[],"status":"failed","error":""}
    audio_dir=CONFIG["transcript"]["whisper_audio_dir"]; os.makedirs(audio_dir, exist_ok=True)
    temp_dir=tempfile.mkdtemp(prefix="whisper_", dir=audio_dir)
    audio_path=""
    try:
        ydl_opts={
            "quiet":True,"format":"bestaudio/best",
            "outtmpl":os.path.join(temp_dir,"audio.%(ext)s"),
            "noplaylist":True,
            "retries":10,"fragment_retries":10,"extractor_retries":10,"file_access_retries":5,
            "socket_timeout":60,
            "sleep_interval":2,"max_sleep_interval":5,
            "http_headers":{
                "User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
                "Accept-Language":"ja,en-US;q=0.9,en;q=0.8",
            },
        }
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info=ydl.extract_info(url, download=True)
            audio_path=ydl.prepare_filename(info)
        duration=0
        with yt_dlp.YoutubeDL({"quiet":True}) as ydl:
            duration=(ydl.extract_info(url, download=False) or {}).get("duration") or 0
        if duration and duration > CONFIG["transcript"]["whisper_max_duration_sec"]:
            out["status"]="timeout"; out["error"]=f"duration_exceeds_limit:{duration}"; return out
        segments, _ = whisper_model.transcribe(audio_path, language=lang_hint, vad_filter=True)
        seg_list=[]; texts=[]
        for s in segments:
            seg_list.append({"start":s.start,"end":s.end,"text":s.text}); texts.append(s.text)
        out["text"]=re.sub(r"\s+"," "," ".join(texts)).strip(); out["segments"]=seg_list; out["status"]="done"
    except subprocess.TimeoutExpired as ex:
        out["status"]="timeout"; out["error"]=str(ex)
    except Exception as ex:
        out["status"]="failed"; out["error"]=str(ex)
    finally:
        shutil.rmtree(temp_dir, ignore_errors=True)
    return out

# Remaining code minimally adapted
whisper_stats={"done":0,"failed":0,"minutes":0.0}; date_excluded_total=0; date_included_total=0

def get_youtube_entries(src):
    global date_excluded_total, date_included_total
    ydl_opt={
        "quiet":True,"extract_flat":"in_playlist","skip_download":True,"ignoreerrors":True,
        "retries":10,"extractor_retries":10,"file_access_retries":5,
        "socket_timeout":60,
        "sleep_interval":2,"max_sleep_interval":5,
        "sleep_interval_requests":1,
        "http_headers":{
            "User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            "Accept-Language":"ja,en-US;q=0.9,en;q=0.8",
        },
    }
    rec=[]; st="ok"; err=""; excluded=0; info=None
    retry_count = CONFIG["collection"].get("yt_retry_count", 3)
    retry_wait = CONFIG["collection"].get("yt_retry_wait_sec", 5)
    for attempt in range(retry_count):
        try:
            with yt_dlp.YoutubeDL(ydl_opt) as ydl:
                info=ydl.extract_info(src["url"], download=False) or {}
            if info.get("entries"):
                break
        except Exception as ex:
            err=str(ex)
            if attempt < retry_count - 1:
                wait_sec = retry_wait * (attempt + 1)  # 指数的バックオフ
                log_check(f"YT retry {attempt+1}/{retry_count} for {src['name']}", f"{err[:80]} | wait {wait_sec}s")
                time.sleep(wait_sec)
            else:
                st="failed"
                return rec, st, err, excluded
    try:
        entries = (info or {}).get("entries") or []
        if "filters" in src:
            filtered = []
            for e in entries:
                t = (e.get("title") or "").lower() + " " + (e.get("description") or "").lower()
                if any(f.lower() in t for f in src["filters"]):
                    filtered.append(e)
            entries = filtered
        for e in entries:
            if not is_within_window(e.get("upload_date",""), CONFIG["collection"]["fetch_window_days"]):
                excluded += 1; continue
            rec.append({"id":e.get("id"),"source_url":e.get("url") or e.get("webpage_url")})
        date_excluded_total += excluded; date_included_total += len(rec)
    except Exception as ex:
        st="failed"; err=str(ex)
    return rec[:src.get("max_items",3)], st, err, excluded


def get_youtube_entries_with_fallback(src):
    """直近2日で0件なら7日まで拡張して再試行"""
    original_window = CONFIG["collection"]["fetch_window_days"]
    rec, st, err, excluded = get_youtube_entries(src)
    if len(rec) == 0 and st == "ok":
        log_check(f"Date window fallback for {src['name']}", "extending 2d -> 7d")
        CONFIG["collection"]["fetch_window_days"] = 7
        try:
            rec, st, err, excluded = get_youtube_entries(src)
        finally:
            CONFIG["collection"]["fetch_window_days"] = original_window
    return rec, st, err, excluded

def pick_caption(subs, auto):
    langs = CONFIG["transcript"]["preferred_langs"]
    # 人手字幕を優先、次に自動字幕
    for grp, name in [(subs or {}, "subtitle"), (auto or {}, "auto_subtitle")]:
        for lang in langs:
            if lang in grp and grp[lang]:
                return grp[lang][0].get("url"), name
    return None, None

def enrich_video(src, vid):
    url = vid.get("source_url")
    if url and not str(url).startswith("http"):
        url = f"https://www.youtube.com/watch?v={url}"
    meta = {
        "source_group": src["source_group"], "source_name": src["name"],
        "source_kind": src["source_kind"], "speaker_label": src["speaker_label"],
        "directness": src["directness"], "url": url, "title": "",
        "published_at": "", "description": "", "thumbnail_url": "",
        "transcript_text": "", "transcript_segments": [],
        "transcript_status": "metadata_only",
        "audio_transcription_status": "skipped", "error_message": ""
    }
    try:
        ydl_meta_opts = {
            "quiet": True, "skip_download": True, "ignoreerrors": True,
            "retries": 10, "extractor_retries": 10, "file_access_retries": 5,
            "socket_timeout": 60,
            "sleep_interval": 2, "max_sleep_interval": 5,
            "http_headers": {
                "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
                "Accept-Language": "ja,en-US;q=0.9,en;q=0.8",
            },
        }
        with yt_dlp.YoutubeDL(ydl_meta_opts) as ydl:
            inf = ydl.extract_info(url, download=False) or {}
        meta.update({
            "title": inf.get("title", ""),
            "published_at": inf.get("upload_date", ""),
            "description": inf.get("description", ""),
            "thumbnail_url": inf.get("thumbnail", "")
        })
        # 字幕取得
        cap_url, cap_type = pick_caption(inf.get("subtitles"), inf.get("automatic_captions"))
        if cap_url:
            if "fmt=" not in cap_url:
                cap_url = cap_url + ("&" if "?" in cap_url else "?") + "fmt=vtt"
            try:
                txt = requests.get(cap_url, timeout=30).text
                parsed = parse_caption(txt)
                if parsed and len(parsed) >= 200:
                    meta["transcript_text"] = parsed
                    meta["transcript_status"] = cap_type
            except Exception as ex:
                meta["error_message"] = f"caption_fetch_failed:{ex}"
        # 字幕が取れなかった or 短すぎる場合 Whisper にフォールバック
        if (not meta["transcript_text"] or len(meta["transcript_text"]) < 200) \
                and CONFIG["transcript"]["transcribe_audio_if_no_subtitle"] \
                and whisper_model is not None:
            meta["audio_transcription_status"] = "running"
            result = transcribe_with_whisper(url, lang_hint="ja")
            meta["audio_transcription_status"] = result["status"]
            if result["status"] == "done" and result["text"]:
                meta["transcript_text"] = result["text"]
                meta["transcript_segments"] = result["segments"]
                meta["transcript_status"] = "whisper"
                whisper_stats["done"] += 1
            else:
                whisper_stats["failed"] += 1
                meta["error_message"] = result.get("error", "")
    except Exception as ex:
        meta["error_message"] = str(ex)
    return meta

bullish_words = ["buy","bullish","upside","outperform","強気","上昇","追い風"]
bearish_words = ["sell","bearish","downside","underperform","弱気","下落"]
warn_words = ["risk","warning","注意","懸念","慎重"]

def fallback_extract(item):
    txt = item.get("transcript_text") or (item.get("title","") + " " + item.get("description",""))
    if not txt.strip():
        return []
    method = "transcript_rule" if item.get("transcript_text") else "metadata_fallback"
    rows = []
    for tk, nm in entity_map.items():
        if re.search(rf"\b{re.escape(tk)}\b|{re.escape(nm)}", txt, re.IGNORECASE):
            low_txt = txt.lower()
            i = low_txt.find(tk.lower())
            if i < 0:
                i = low_txt.find(nm.lower())
            if i < 0:
                i = 0
            ctx = txt[max(0, i-300):i+300]
            low = ctx.lower()
            stance = "neutral"
            if any(w in low for w in bearish_words): stance = "bearish"
            elif any(w in low for w in bullish_words): stance = "bullish"
            elif any(w in low for w in warn_words): stance = "warning"
            rows.append({
                "statement_id": f"st_{len(rows)+1}_{abs(hash(item['url']))%99999}",
                "source_group": item["source_group"], "source_name": item["source_name"],
                "source_kind": item["source_kind"], "speaker_label": item["speaker_label"],
                "directness": item["directness"], "title": item["title"],
                "source_url": item["url"], "published_at": item["published_at"],
                "transcript_status": item["transcript_status"],
                "timestamp_start": "", "timestamp_end": "",
                "ticker": tk, "company_name_en": nm,
                "company_name_kana": company_kana_map.get(nm, "推定読み"),
                "company_display_name": f"{nm}({company_kana_map.get(nm,'推定読み')})",
                "asset_type": "stock", "validated": False,
                "topic": "簡易抽出", "stance": stance, "stance_strength": 0.45,
                "time_horizon": "unclear",
                "thesis_summary": ctx[:120], "reason_summary": ctx[:120],
                "risk_summary": ctx[:80], "evidence_quote": ctx[:100],
                "confidence": "medium" if method == "transcript_rule" else "low",
                "extraction_method": method, "mention_context": ctx
            })
    return rows

all_items = []
source_rows = []

# YouTube系（cramer, dalio）
for grp_key in ["cramer", "dalio"]:
    for s in CONFIG["sources"][grp_key]:
        if not s.get("enabled", True):
            continue
        ents, st, err, excluded = get_youtube_entries_with_fallback(s)
        vids = [enrich_video(s, v) for v in ents]
        all_items.extend(vids)
        source_rows.append({
            "source_group": s["source_group"], "source_name": s["name"],
            "source_url": s["url"], "status": st, "error_message": err,
            "fetched_items_count": len(vids), "excluded_by_date_count": excluded,
            "items_with_transcript": sum(1 for x in vids if x["transcript_text"]),
            "items_with_audio_transcript": sum(1 for x in vids if x["audio_transcription_status"] == "done")
        })

# Pivot ポッドキャスト
for s in CONFIG["sources"]["galloway"]:
    if not s.get("enabled", True):
        continue
    rec = []; st = "ok"; err = ""; excluded = 0
    try:
        rss = s.get("rss_url")
        if not rss:
            q = requests.get("https://itunes.apple.com/search",
                             params={"media": "podcast", "term": s["podcast_search_query"], "limit": 1},
                             timeout=20).json()
            rss = (q.get("results") or [{}])[0].get("feedUrl", "")
        d = feedparser.parse(rss)
        for e in d.entries[:s.get("max_items", 3)]:
            pub = e.get("published", "")
            if not is_within_window(pub, CONFIG["collection"]["fetch_window_days"]):
                excluded += 1
                continue
            audio_url = ""
            for enc in e.get("enclosures", []):
                if (enc.get("type") or "").startswith("audio/"):
                    audio_url = enc.get("href") or enc.get("url") or ""
                    break
            transcript_text = ""
            transcript_segments = []
            transcript_status = "metadata_only"
            audio_status = "skipped"
            if audio_url and CONFIG["transcript"]["transcribe_audio_if_no_subtitle"] and whisper_model is not None:
                audio_status = "running"
                result = transcribe_with_whisper(audio_url, lang_hint="en")
                audio_status = result["status"]
                if result["status"] == "done" and result["text"]:
                    transcript_text = result["text"]
                    transcript_segments = result["segments"]
                    transcript_status = "whisper"
                    whisper_stats["done"] += 1
                else:
                    whisper_stats["failed"] += 1
            if not transcript_text:
                transcript_text = e.get("summary", "")
                if transcript_text:
                    transcript_status = "description_fallback"
            rec.append({
                "source_group": s["source_group"], "source_name": s["name"],
                "source_kind": s["source_kind"], "speaker_label": s["speaker_label"],
                "directness": s["directness"], "title": e.get("title", ""),
                "url": e.get("link", rss), "published_at": pub,
                "description": e.get("summary", ""), "thumbnail_url": "",
                "transcript_text": transcript_text,
                "transcript_segments": transcript_segments,
                "transcript_status": transcript_status,
                "audio_transcription_status": audio_status, "error_message": ""
            })
        date_excluded_total += excluded
    except Exception as ex:
        st = "failed"; err = str(ex)
    all_items.extend(rec)
    source_rows.append({
        "source_group": s["source_group"], "source_name": s["name"],
        "source_url": s.get("rss_url") or s.get("podcast_search_query"),
        "status": st, "error_message": err,
        "fetched_items_count": len(rec), "excluded_by_date_count": excluded,
        "items_with_transcript": sum(1 for x in rec if x["transcript_text"]),
        "items_with_audio_transcript": sum(1 for x in rec if x["audio_transcription_status"] == "done")
    })

statement_units = []
for it in all_items:
    statement_units.extend(fallback_extract(it))

mentions = [{
    "ticker": r["ticker"], "company_name_en": r["company_name_en"],
    "company_name_kana": r["company_name_kana"], "asset_type": r["asset_type"],
    "validated": r["validated"], "source_group": r["source_group"],
    "source_name": r["source_name"], "source_url": r["source_url"],
    "mention_context": r["mention_context"], "stance": r["stance"],
    "confidence": r["confidence"]
} for r in statement_units]

json.dump(all_items, open(raw_json_path, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
pd.DataFrame(source_rows).to_csv(source_csv_path, index=False)
pd.DataFrame(mentions).to_csv(mentions_csv_path, index=False)
pd.DataFrame(statement_units).to_csv(statement_csv_path, index=False)

by_person = []
if statement_units:
    su_df = pd.DataFrame(statement_units)
    for (tk, nm), grp in su_df.groupby(["ticker", "company_name_en"]):
        row = {"ticker": tk, "company_name_en": nm,
               "company_name_kana": grp["company_name_kana"].iloc[0],
               "asset_type": grp["asset_type"].iloc[0], "validated": False}
        for sg, col in [("Jim Cramer", "cramer"), ("Ray Dalio", "dalio"),
                        ("Scott Galloway / Pivot", "galloway")]:
            g = grp[grp["source_group"] == sg]
            row[f"mentioned_by_{col}"] = len(g) > 0
            row[f"{col}_stance"] = g["stance"].mode().iloc[0] if len(g) > 0 else "直接言及なし"
            row[f"{col}_reason"] = g["reason_summary"].iloc[0] if len(g) > 0 else ""
        row["combined_note"] = "要確認"
        by_person.append(row)
pd.DataFrame(by_person).to_csv(by_person_csv_path, index=False)

su = pd.DataFrame(statement_units) if statement_units else pd.DataFrame(columns=["source_group", "thesis_summary"])
def summary_for(group_name):
    g = su[su["source_group"] == group_name] if "source_group" in su.columns else pd.DataFrame()
    pts = g["thesis_summary"].dropna().head(5).tolist() if "thesis_summary" in g.columns else []
    if not pts: pts = ["発言データ不足"]
    return {
        "count": len(g),
        "transcript_count": sum(1 for x in all_items if x["source_group"] == group_name and x["transcript_text"]),
        "point": pts
    }
sc = summary_for("Jim Cramer")
sd = summary_for("Ray Dalio")
sg_sum = summary_for("Scott Galloway / Pivot")
common_points = ["本文ベースで評価し、タイトル依存を回避", "リスク記述を明示", "確信度をlow/mediumで管理"]
diff_points = ["Cramer系は個別株文脈中心", "Dalio系はマクロ中心", "Pivot系はプラットフォーム競争中心"]

checks = {
    "raw_sources_json": os.path.exists(raw_json_path),
    "source_check_csv": os.path.exists(source_csv_path),
    "extracted_mentions_csv": os.path.exists(mentions_csv_path),
    "statement_units_csv": os.path.exists(statement_csv_path),
    "mentioned_by_person_csv": os.path.exists(by_person_csv_path),
    "any_items_collected": len(all_items) > 0,
    "no_json_artifact_in_transcript": all(
        '"utf8":' not in (i.get("transcript_text") or "")[:5000] for i in all_items
    )
}
quality_reasons = [f"{k} failed" for k, v in checks.items() if not v]
quality_passed = len(quality_reasons) == 0
json.dump({"passed": quality_passed, "reasons": quality_reasons, "checks": checks},
          open(quality_json_path, "w", encoding="utf-8"), ensure_ascii=False, indent=2)

html = f"""<!doctype html><html><head><meta charset='utf-8'><style>
@page {{ size: A4 landscape; margin: 10mm; }}
body {{ font-family: sans-serif; font-size: 10px; }}
table {{ width: 100%; border-collapse: collapse; }}
td, th {{ border: 1px solid #ccc; padding: 4px; }}
</style></head><body>
<h1>米国株AI投資調査レポート</h1><p>作成日: {today}</p>
<h2>1ページ目 要約</h2>
<p>収集件数: {len(all_items)}件 / 発言抽出: {len(statement_units)}件 / 言及銘柄: {len(by_person)}件</p>
<h2>2ページ目 人物別サマリー</h2>
<p>Jim Cramer系: {sc['count']}件 / transcript {sc['transcript_count']}件</p>
<ul>{''.join(f'<li>{x}</li>' for x in sc['point'])}</ul>
<p>Ray Dalio系: {sd['count']}件 / transcript {sd['transcript_count']}件</p>
<ul>{''.join(f'<li>{x}</li>' for x in sd['point'])}</ul>
<p>Scott Galloway/Pivot系: {sg_sum['count']}件 / transcript {sg_sum['transcript_count']}件</p>
<ul>{''.join(f'<li>{x}</li>' for x in sg_sum['point'])}</ul>
<h2>3ページ目 銘柄横並び比較</h2>{pd.DataFrame(by_person).to_html(index=False) if by_person else '<p>データなし</p>'}
<h2>4ページ目 共通点と相違点</h2>
<ul>{''.join(f'<li>{x}</li>' for x in common_points + diff_points)}</ul>
<h2>5ページ目 全銘柄</h2>{pd.DataFrame(mentions).head(100).to_html(index=False) if mentions else '<p>データなし</p>'}
<h2>6ページ目以降 詳細</h2>{pd.DataFrame(statement_units).head(200).to_html(index=False) if statement_units else '<p>データなし</p>'}
</body></html>"""
open(html_path, "w", encoding="utf-8").write(html)

try:
    HTML(string=html, base_url=DRIVE_ROOT).write_pdf(pdf_path)
except Exception as ex:
    log_check("PDF generation error", str(ex))

log_check("Date filter window days", CONFIG["collection"]["fetch_window_days"])
log_check("Items excluded by date filter", date_excluded_total)
log_check("Items within date window", date_included_total)
log_check("Whisper enabled", CONFIG["transcript"]["transcribe_audio_if_no_subtitle"])
log_check("Whisper model loaded", whisper_model is not None)
log_check("Items transcribed by Whisper", whisper_stats["done"])
log_check("Whisper failures", whisper_stats["failed"])
log_check("Total items collected", len(all_items))
log_check("Statement units generated", len(statement_units))
log_check("Mentioned tickers", len(by_person))
log_check("Quality gate passed", "yes" if quality_passed else "no")
log_check("PDF generated", "yes" if os.path.exists(pdf_path) else "no")
log_check("PDF path", pdf_path)
log_check("PDF size KB", round(os.path.getsize(pdf_path)/1024, 2) if os.path.exists(pdf_path) else 0)

email_sent = False
if (not DRY_RUN) and SEND_EMAIL and quality_passed and os.path.exists(pdf_path) \
        and EMAIL_TO and SMTP_USER and SMTP_PASSWORD:
    msg = EmailMessage()
    msg["Subject"] = f"米国株AIレポート {today}"
    msg["From"] = SMTP_USER; msg["To"] = EMAIL_TO
    msg.set_content("PDFを添付します。")
    with open(pdf_path, "rb") as f:
        msg.add_attachment(f.read(), maintype="application", subtype="pdf",
                           filename=os.path.basename(pdf_path))
    with smtplib.SMTP("smtp.gmail.com", 587, timeout=30) as smtp:
        smtp.starttls(); smtp.login(SMTP_USER, SMTP_PASSWORD); smtp.send_message(msg)
    email_sent = True
log_check("Email sent", "yes" if email_sent else "no")


## 次にやること

1回目が成功したら、次は検索クエリや対象銘柄を増やします。  
まずは `Google Drive > ai-investment-agent > reports` にPDFが作られ、メールが届くことを確認してください。

In [ ]:
# セル4：PDF存在確認セル
import os

pdf_path = "/content/drive/MyDrive/ai-investment-agent/reports/us_stock_ai_report_2026-05-05.pdf"

print("PDF exists:", os.path.exists(pdf_path))
if os.path.exists(pdf_path):
    print("PDF size KB:", os.path.getsize(pdf_path) / 1024)


In [ ]:
# セル5：最新reports確認セル
!ls -lh /content/drive/MyDrive/ai-investment-agent/reports


In [ ]:
# セル6：メール送信テストセル（確認入力あり）
import os

test_pdf_path = input("送信テストするPDFパスを入力してください: ").strip()
confirm = input("このPDFをメール送信テストしますか？ yes/no: ").strip().lower()

if confirm != "yes":
    print("送信をキャンセルしました。")
elif not os.path.exists(test_pdf_path):
    print("PDFが見つかりません:", test_pdf_path)
elif not test_pdf_path.lower().endswith('.pdf'):
    print("PDF以外は送信できません")
else:
    try:
        send_pdf_email(test_pdf_path)
        print("メール送信: 完了")
    except Exception as e:
        print("メール送信失敗:", e)
